<a href="https://colab.research.google.com/github/asve06/act1_2p_si_eda_2_25_vega/blob/main/code/act2_2p_si_tech_2_25_vega.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Women Perfume Recommendation Based on Note Similarities Project**
Ashley Vega   
*Task:* Recommend similar perfumes based on their descriptions.   
*Description:*  The goal of this project is to build a recommendation system that suggests perfumes with similar characteristics or notes by analyzing text data, focusing only on women’s fragrances. This idea explores how machine learning can be used to understand preferences and find connections between products.  
*Task Type:* Classification.  
*Algorithm:* K-Nearest Neighbors (K-NN).

# **Activity 2: Categorical data encoding and feature scaling techniques**

In [20]:
import pandas as pd

In [21]:
df_women= pd.read_csv("/content/df_women.csv")

In [22]:
#Function to count how many scent notes appear in both perfumes.

def note_overlap_count(text1, text2):
    return len(set(text1.split()) & set(text2.split()))

#Model A Simple Baseline (no encoding)

In [23]:
# Jaccard Similarity Coefficient Calculation(Intersection / Union).

def simple_similarity(a, b):
    # Preparation
    set_a = set(a.split())
    set_b = set(b.split())

    # Handle Empty Sets
    if len(set_a) == 0 or len(set_b) == 0:
        return 0

    # Jaccard Calculation
    return len(set_a & set_b) / len(set_a | set_b)

In [24]:
# Perfume recommendations generation using Jaccard Similarity on scent profiles.
def basic_recommendation(name, n=5):
  name = name.strip()

  matches = df_women[df_women['Perfume'].str.contains(name, case=False)]
  if matches.empty:
    print(f" - - Perfume '{name}' not found. - -")
    return

  idx= matches.index[0]

  # Base Profile creation - Combination of all scent fields into a single string.
  base_features = df_women.loc[idx, [
      "Top","Middle","Base",
      "mainaccord1","mainaccord2","mainaccord3","mainaccord4","mainaccord5"
  ]].str.cat(sep=" ")

  # Store the results
  similarities = []

  # Iterate and Score - Compare the base perfume against every other perfume in the dataset.
  for i in range(len(df_women)):
      if i == idx:
          continue # Skip comparing the perfume with itself.

      # Feature profile creation for the current perfume (i).
      features = df_women.loc[i, [
          "Top","Middle","Base",
          "mainaccord1","mainaccord2","mainaccord3","mainaccord4","mainaccord5"
      ]].str.cat(sep=" ")

      # Jaccard similarity score calculation
      score = simple_similarity(base_features, features)

      # NoteOverlap
      overlap = note_overlap_count(base_features, features)

      similarities.append((i, score, overlap))

  # Sort and Select Top 'n'- By score in descending order (highest score first) and keep the top 'n'.
  similarities = sorted(similarities, key=lambda x: x[1], reverse=True)[:n]

  # Show results
  print(f"\n🔍 Perfume basic recommendations for: '{df_women.loc[idx,'Perfume'].title()}' "
        f"by {df_women.loc[idx,'Brand'].title()}\n")

  for idx_sim, score, overlap in similarities:
        p = df_women.loc[idx_sim]
        print(f" - {p['Perfume'].title()} | Brand: {p['Brand'].title()} "
              f"| Similarity: {round(score*100,2)}% | NoteOverlap: {overlap}")

#Model B Encoding Techniques (TD-IDF)

In [25]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

In [26]:
# Unified feature column creation with all scent notes
df_women["features"] = (
    df_women["Top"] + " " +
    df_women["Middle"] + " " +
    df_women["Base"] + " " +
    df_women["mainaccord1"] + " " +
    df_women["mainaccord2"] + " " +
    df_women["mainaccord3"] + " " +
    df_women["mainaccord4"] + " " +
    df_women["mainaccord5"]
)

# TF-IDF Vectorization - features' convertion from text data to a numerical matrix
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(df_women["features"])

# KNN Model using Cosine Distance - measure similarity between text documents
knn = NearestNeighbors(metric='cosine')
knn.fit(tfidf_matrix)

NearestNeighbors(metric='cosine')

In [27]:
def perfume_recommendation(name, n=5):
    name = name.lower().strip()

    matches = df_women[df_women["Perfume"].str.contains(name, case=False)]
    if matches.empty:
        print(f"❌ No existe el perfume '{name}' en el dataset.")
        return

    # Get the index of the matching perfume.
    idx = matches.index[0]
    base_perfume = df_women.iloc[idx]

    # Find the n+1 nearest neighbors based on the TF-IDF vectors including the base perfume itself
    distances, indices = knn.kneighbors(tfidf_matrix[idx], n_neighbors=n+1)

    print(f"\n🔍 Perfume recommendations for: '{base_perfume['Perfume'].title()}' "
          f"by {base_perfume['Brand'].title()}\n")

    # Iterate through the results, starting from index 1 (to skip the base perfume).
    for i in range(1, n+1):
        sim_idx = indices[0][i]
        perfume = df_women.iloc[sim_idx]

        similarity = (1 - distances[0][i]) * 100

        base_text = df_women.loc[idx, ["Top","Middle","Base"]].str.cat(sep=" ")
        comp_text = df_women.loc[sim_idx, ["Top","Middle","Base"]].str.cat(sep=" ")

        overlap = note_overlap_count(base_text, comp_text)

        print(f"- {perfume['Perfume'].title()} | Brand: {perfume['Brand'].title()} "
              f"| Similarity: {similarity:.2f}% | NoteOverlap: {overlap}")

In [28]:
# Sample of perfume names available in the datasetsample of perfume names available in the dataset
df_women["Perfume"].sample(15).str.title()

,Perfume
9988,Cyane
8313,Angel-Eau-Sucree-2016
11217,Impossible-Bouquet-Eau-Moheli
2992,Treselle
2459,Acqua-Di-Gioia-Jasmine
3057,Bambu-For-Her
9319,Intense-For-Women
587,Volare-Magnolia
1268,B-U-Golden-Kiss
1382,Baiser-Vole-Parfum


In [29]:
def compare_models(name):
    print("Model A - Simple Baseline (no encoding)")
    basic_recommendation(name)

    print("\n- - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - -")
    print("\nMODEL B: TF-IDF + KNN")
    perfume_recommendation(name)

compare_models("la-petite-robe-noire")

Model A - Simple Baseline (no encoding)

🔍 Perfume basic recommendations for: 'La-Petite-Robe-Noire-2' by Guerlain

 - Insolence | Brand: Guerlain | Similarity: 53.85% | NoteOverlap: 14
 - Mademoiselle-Guerlain | Brand: Guerlain | Similarity: 40.74% | NoteOverlap: 11
 - Amyi-Iv | Brand: Amyi | Similarity: 38.46% | NoteOverlap: 10
 - Insolence-Shimmering-Edition | Brand: Guerlain | Similarity: 33.33% | NoteOverlap: 10
 - N01-De-Chanel-L-Eau-Rouge | Brand: Chanel | Similarity: 30.77% | NoteOverlap: 8

- - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - -

MODEL B: TF-IDF + KNN

🔍 Perfume recommendations for: 'La-Petite-Robe-Noire-2' by Guerlain

- Mademoiselle-Guerlain | Brand: Guerlain | Similarity: 63.80% | NoteOverlap: 7
- Insolence-Limited-Edition | Brand: Guerlain | Similarity: 55.93% | NoteOverlap: 4
- Insolence | Brand: Guerlain | Similarity: 54.66% | NoteOverlap: 9
- Married | Brand: Franck-Boclet | Similarity: 49.51% | NoteOverlap: 5
- Insolence-Shimmering-Edit

In [30]:
compare_models("Flowerbomb")

Model A - Simple Baseline (no encoding)

🔍 Perfume basic recommendations for: 'Flowerbomb-Bomblicious' by Viktor-Rolf

 - Diesel-Fuel-For-Life-Cologne-For-Women | Brand: Diesel | Similarity: 50.0% | NoteOverlap: 11
 - Ines-De-La-Fressange-2004 | Brand: Ines-De-La-Fressange | Similarity: 46.15% | NoteOverlap: 12
 - Bohemian-Romance-Eau-De-Toilette | Brand: Betty-Barclay | Similarity: 44.0% | NoteOverlap: 11
 - Soie-Rouge | Brand: Avon | Similarity: 44.0% | NoteOverlap: 11
 - Tiffany-Co | Brand: Tiffany | Similarity: 41.67% | NoteOverlap: 10

- - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - -

MODEL B: TF-IDF + KNN

🔍 Perfume recommendations for: 'Flowerbomb-Bomblicious' by Viktor-Rolf

- Cool-Water-Sea-Rose-Exotic-Summer | Brand: Davidoff | Similarity: 73.95% | NoteOverlap: 2
- La-Mia-Perla-Nera | Brand: La-Perla | Similarity: 69.31% | NoteOverlap: 5
- Coquette | Brand: Faberlic | Similarity: 64.76% | NoteOverlap: 5
- Rosa-Damascena | Brand: Granado | Similarity: 64